## Regression Problem with ANN

**EstimatedSalary as target variable**

In [159]:
import pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [160]:
df = pd.read_csv('data/customer_churn_data.csv')
df.head()

# EstimatedSalary as target variable

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [161]:
# Drop unnecessary columns
df.drop(columns=['RowNumber', 'CustomerId', 'Surname'], inplace=True)

In [162]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### Encoding Geography

In [163]:
geography_ohe = OneHotEncoder(drop='first', sparse_output=False)
geography_ohe_data = geography_ohe.fit_transform(df[['Geography']])
geography_ohe_data

array([[0., 0.],
       [0., 1.],
       [0., 0.],
       ...,
       [0., 0.],
       [1., 0.],
       [0., 0.]], shape=(10000, 2))

In [164]:
geography_ohe.get_feature_names_out()

array(['Geography_Germany', 'Geography_Spain'], dtype=object)

In [165]:
geography_ohe_df = pd.DataFrame(geography_ohe_data, columns=geography_ohe.get_feature_names_out())
geography_ohe_df.head()

,Geography_Germany,Geography_Spain
0,0.0,0.0
1,0.0,1.0
2,0.0,0.0
3,0.0,0.0
4,0.0,1.0


### Encoding Gender

In [166]:
gender_ohe = OneHotEncoder(drop='first', sparse_output=False)
gender_ohe_data = gender_ohe.fit_transform(df[['Gender']])
gender_ohe_data

array([[0.],
       [0.],
       [0.],
       ...,
       [0.],
       [1.],
       [0.]], shape=(10000, 1))

In [167]:
gender_ohe_df = pd.DataFrame(gender_ohe_data, columns=gender_ohe.get_feature_names_out())
gender_ohe_df.head()

,Gender_Male
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0


#### Updating original dataframe with encoded columns

In [168]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [169]:
df.drop(columns=['Geography', 'Gender'], inplace=True)
df.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,42,2,0.00,1,1,1,101348.88,1
1,608,41,1,83807.86,1,0,1,112542.58,0
2,502,42,8,159660.80,3,1,0,113931.57,1
3,699,39,1,0.00,2,0,0,93826.63,0
4,850,43,2,125510.82,1,1,1,79084.10,0


In [170]:
df = pd.concat([df, gender_ohe_df, geography_ohe_df], axis=1)
df.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Gender_Male,Geography_Germany,Geography_Spain
0,619,42,2,0.00,1,1,1,101348.88,1,0.0,0.0,0.0
1,608,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,42,8,159660.80,3,1,0,113931.57,1,0.0,0.0,0.0
3,699,39,1,0.00,2,0,0,93826.63,0,0.0,0.0,0.0
4,850,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


### Splitting data for train and test

In [171]:
X = df.drop('EstimatedSalary', axis=1)
y = df['EstimatedSalary']

In [172]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Standardizing the data

In [173]:
# fit transform on train
# transform on test

# Both are for X

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Regression ANN

In [174]:
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [175]:
X_train.shape[1]

11

In [176]:
X_train_scaled[0].shape

(11,)

In [177]:
model = Sequential([
    Dense(64, activation='relu', input_shape=X_train_scaled[0].shape),
    Dense(32, activation='relu'),
    Dense(1) # Output layer for regression
])

# Compile Model
model.compile(optimizer='adam', loss='mean_absolute_error', metrics=['mae'])

model.summary()

/workspaces/ANN-Customer-Churn/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_24 (Dense)                │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

In [178]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime('%Y%M%D-%H%M%S')
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [179]:
# Set Up Early Stopping
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [180]:
# Train model
history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=100,
    callbacks=[early_stopping_callback, tensorboard_callback]
)

Epoch 1/100


250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 100395.6562 - mae: 100395.6562 - val_loss: 98584.0703 - val_mae: 98584.0703
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 99874.0859 - mae: 99874.0859 - val_loss: 97521.5547 - val_mae: 97521.5547
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 98013.3047 - mae: 98013.3047 - val_loss: 94772.1562 - val_mae: 94772.1562
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 94263.7500 - mae: 94263.7500 - val_loss: 90028.7656 - val_mae: 90028.7656
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 88586.5469 - mae: 88586.5469 - val_loss: 83554.6719 - val_mae: 83554.6719
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 81392.3047 - mae: 81392.3047 - val_loss: 76031.7031 - val_mae: 76031.7031
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 73570.5234 - mae: 73570.5234 - val_loss: 68565.5312 - val_mae: 68565.5312
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 6621

In [181]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [182]:
%tensorboard --logdir regressionlogs/fit

Reusing TensorBoard on port 6006 (pid 28346), started 0:07:12 ago. (Use '!kill 28346' to kill it.)

In [183]:
test_loss, test_mae = model.evaluate(X_test_scaled, y_test)
print(f"Test MAE: {test_mae}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 50326.3984 - mae: 50326.3984  
Test MAE: 50326.3984375


In [184]:
model.save('model_artifacts/regression_model.h5')